`conda activate r_4.3.3`

In [ ]:
library(ArchR)
set.seed(1)

Loading required package: ggplot2

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats

In [ ]:
library(parallel)

In [ ]:
# Configure
addArchRThreads(threads = 32)
addArchRGenome('hg38')

In [ ]:
proj <- loadArchRProject('./ArchR_malignant_FINAL/')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [ ]:
# SVD, Clustering, UMAP
proj <- addIterativeLSI(ArchRProj = proj, useMatrix = "TileMatrix",
                       name = "IterativeLSI", force=TRUE)

Checking Inputs...

ArchR logging to : ArchRLogs/ArchR-addIterativeLSI-3f9ce54060e437-Date-2024-08-22_Time-13-56-28.067311.log
If there is an issue, please report to github with logFile!

2024-08-22 13:56:29.607439 : Computing Total Across All Features, 0.009 mins elapsed.

2024-08-22 13:56:42.771481 : Computing Top Features, 0.229 mins elapsed.

###########
2024-08-22 13:56:43.741347 : Running LSI (1 of 2) on Top Features, 0.245 mins elapsed.
###########

2024-08-22 13:56:43.82685 : Sampling Cells (N = 10013) for Estimated LSI, 0.246 mins elapsed.

2024-08-22 13:56:43.829701 : Creating Sampled Partial Matrix, 0.247 mins elapsed.

2024-08-22 13:56:57.121918 : Computing Estimated LSI (projectAll = FALSE), 0.468 mins elapsed.

2024-08-22 13:57:10.521924 : Identifying Clusters, 0.691 mins elapsed.

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
2024-08-22 13:57:24.379517 : Identified 6 Clusters, 0.922 mins elapsed.

2024-08-22 13:57:24.398614 : Saving LSI Iteration, 0.


************************************************************
2024-08-22 13:57:32.33926 : ERROR Found in .saveIteration for  
LogFile = ArchRLogs/ArchR-addIterativeLSI-3f9ce54060e437-Date-2024-08-22_Time-13-56-28.067311.log

<simpleError in g$grobs[[legend]]: no such index at level 2
>

************************************************************



2024-08-22 13:57:32.345893 : Creating Cluster Matrix on the total Group Features, 1.055 mins elapsed.

2024-08-22 13:57:45.947261 : Computing Variable Features, 1.282 mins elapsed.

###########
2024-08-22 13:57:46.102574 : Running LSI (2 of 2) on Variable Features, 1.284 mins elapsed.
###########

2024-08-22 13:57:46.141373 : Creating Partial Matrix, 1.285 mins elapsed.

2024-08-22 13:58:23.570467 : Computing LSI, 1.909 mins elapsed.

2024-08-22 13:59:06.872426 : Finished Running IterativeLSI, 2.631 mins elapsed.



In [ ]:
# Gene scores with selected features
# Artificial black list to exclude all non variable features
chrs <- getChromSizes(proj)
var_features <- proj@reducedDims[["IterativeLSI"]]$LSIFeatures
var_features_gr <- GRanges(var_features$seqnames, IRanges(var_features$start, var_features$start + 500))
blacklist <- setdiff(chrs, var_features_gr)
proj <- addGeneScoreMatrix(proj, matrixName='GeneScoreMatrix', force=TRUE, blacklist=blacklist)

ArchR logging to : ArchRLogs/ArchR-addGeneScoreMatrix-3f9ce527a8227-Date-2024-08-22_Time-13-59-07.034844.log
If there is an issue, please report to github with logFile!

2024-08-22 13:59:07.346603 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGeneScoreMatrix-3f9ce527a8227-Date-2024-08-22_Time-13-59-07.034844.log



In [ ]:
# Peaks using NFR fragments
proj <- addClusters(input = proj, reducedDims = "IterativeLSI", force = TRUE)
proj <- addGroupCoverages(proj, maxFragmentLength=147, force = TRUE)

ArchR logging to : ArchRLogs/ArchR-addClusters-2adf55a04cbb-Date-2024-08-22_Time-14-20-40.539781.log
If there is an issue, please report to github with logFile!

Overriding previous entry for Clusters

2024-08-22 14:20:44.882398 : Running Seurats FindClusters (Stuart et al. Cell 2019), 0.068 mins elapsed.

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Computing nearest neighbor graph

Computing SNN



Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 83921
Number of edges: 3096803

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.9444
Number of communities: 49
Elapsed time: 15 seconds


20 singletons identified. 29 final clusters.

2024-08-22 14:22:28.673632 : Testing Biased Clusters, 1.798 mins elapsed.

2024-08-22 14:22:29.219011 : Testing Outlier Clusters, 1.807 mins elapsed.

2024-08-22 14:22:29.226612 : Identified more clusters than maxClusters allowed (n = 0). Merging clusters to maxClusters (n = 25).
If this is not desired set maxClusters = NULL!, 1.808 mins elapsed.

2024-08-22 14:22:29.288585 : Assigning Cluster Names to 25 Clusters, 1.809 mins elapsed.

2024-08-22 14:22:29.744345 : Finished addClusters, 1.816 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-2adf57e38238-Date-2024-08-22_Time-14-22-29.750371.log
If there is an issue, please report to github with logFile!

C1 (1 of 25) : CellGroups N = 2

C2 (2 of 25) : CellGroups N = 2

C3 (3 of 25) : CellGroups N = 2

C4 (4 of 25) : CellGroups N = 2

C5 (5 of 25) : CellGroups N = 2

C6 (6 of 25) : CellGroups N = 2

C7 (7 of 25) : CellGroups N = 2

C8 (8 of 25) : CellGroups N = 3

C9 (9 of 2

In [ ]:
pathToMacs2 <- '/hpc/pmc_stunnenberg/cruiz/miniconda3/envs/python2/bin/macs2'
proj <- addReproduciblePeakSet(proj, pathToMacs2 = pathToMacs2, plot = FALSE)
# Counts
proj <- addPeakMatrix(proj, maxFragmentLength=147, ceiling=10^9)

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-2adf6e0eef78-Date-2024-08-22_Time-15-36-38.114949.log
If there is an issue, please report to github with logFile!

Calling Peaks with Macs2

2024-08-22 15:36:38.349465 : Peak Calling Parameters!, 0.004 mins elapsed.



    Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
C1     C1   3815        551           2   51  500   150000
C2     C2     37         34           2   21   24    17000
C3     C3   1677        565           2   65  500   150000
C4     C4   1831        540           2   40  500   150000
C5     C5   2076        540           2   40  500   150000
C6     C6    882        540           2   40  500   150000
C7     C7   1841        540           2   40  500   150000
C8     C8   1140        684           3   89  500   150000
C9     C9    924        540           2   40  500   150000
C10   C10  16809        542           2   42  500   150000
C11   C11    936        540           2   40  500   150000
C12   C12   2541       1917           5  230  463   150000
C13   C13  12543       1720           5  109  500   150000
C14   C14    817        677           3   43  500   150000
C15   C15   1303        540           2   40  500   150000
C16   C16   5600       1365           5   79  500   1500

2024-08-22 15:36:38.361624 : Batching Peak Calls!, 0.004 mins elapsed.

2024-08-22 15:36:38.392397 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-08-22 16:04:50.049873 : Identifying Reproducible Peaks!, 28.199 mins elapsed.

2024-08-22 16:05:45.677058 : Creating Union Peak Set!, 29.126 mins elapsed.

Converged after 10 iterations!

2024-08-22 16:05:52.310374 : Finished Creating Union Peak Set (377577)!, 29.237 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-2adf2d4fc40e-Date-2024-08-22_Time-16-05-52.317007.log
If there is an issue, please report to github with logFile!

2024-08-22 16:05:52.749644 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-2adf2d4fc40e-Date-2024-08-22_Time-16-05-52.317007.log



In [ ]:
meta <- readRDS('metadata_mp.rds')

# Define the pattern and replacement
pattern <- "(.*)_(.*)$"
replacement <- "\\1#\\2"

# Apply the pattern and replacement using sub
rownames(meta) <- sub(pattern, replacement, rownames(meta))
meta

,total,duplicate,chimeric,unmapped,lowmapq,mitochondrial,nonprimary,passed_filters,is__cell_barcode,excluded_reason,⋯,louvain_1_3,louvain_1_5,louvain_1_8,louvain_2,louvain_2_3,louvain_2_5,louvain_2_8,louvain_3,iCNV,MP
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,⋯,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
atac_T19-91014#AAACGAAAGACCCTAT-1,37892,15065,1,438,3127,45,5,19211,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4
atac_T19-91014#AAACGAAAGTGTAATG-1,27213,9888,2,323,2030,71,9,14890,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4
atac_T19-91014#AAACGAACAAATGCTC-1,32533,8763,0,352,2536,206,14,20662,1,0,⋯,1,1,20,16,33,15,34,33,Not measured,MP 3
atac_T19-91014#AAACGAACACGCCGAT-1,17835,5161,3,213,1943,74,16,10425,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4
atac_T19-91014#AAACGAAGTACGGAGT-1,66244,12557,6,786,6822,55,33,45985,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4
atac_T19-91014#AAACGAAGTATCCCTC-1,20966,4675,1,164,1312,11,0,14803,1,0,⋯,1,1,20,16,33,15,34,33,Not measured,MP 3
atac_T19-91014#AAACGAAGTATCTCAG-1,40695,10281,2,501,4589,47,57,25218,1,0,⋯,17,15,18,20,24,22,24,31,Not measured,MP 4
atac_T19-91014#AAACGAAGTCAGACGA-1,10269,1979,1,139,857,2,2,7289,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4
atac_T19-91014#AAACGAAGTGAAACAT-1,24766,10011,1,283,2095,210,10,12156,1,0,⋯,16,18,19,18,26,18,27,21,Not measured,MP 4


In [ ]:
# Step 1: Create a new column for rownames
meta$rownames_col <- rownames(meta)

cellsAtlas <- proj$cellNames
# Step 2: Subset the dataframe based on the desired order
reordered_meta <- meta[match(cellsAtlas, meta$rownames_col), ]

# Step 3: Remove the column containing rownames
reordered_meta$rownames_col <- NULL

reordered_meta

,total,duplicate,chimeric,unmapped,lowmapq,mitochondrial,nonprimary,passed_filters,is__cell_barcode,excluded_reason,⋯,louvain_1_3,louvain_1_5,louvain_1_8,louvain_2,louvain_2_3,louvain_2_5,louvain_2_8,louvain_3,iCNV,MP
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,⋯,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
atac_MUV1#CCCTAACCATGGAGGT-1,124697,4651,8,2051,21081,73,208,96625,1,0,⋯,23,17,17,23,28,21,29,37,Not measured,MP 2
atac_MUV1#GGGTCTGTCGGAGTTT-1,122083,4803,11,2213,22104,34,186,92732,1,0,⋯,23,17,17,23,28,21,29,37,Not measured,MP 2
atac_MUV1#GTGTCCTGTGAGTCGA-1,120397,5377,8,2192,21577,38,117,91088,1,0,⋯,23,17,17,23,13,21,17,37,Not measured,MP 2
atac_MUV1#AGCTGATCATGTATCG-1,112203,6126,16,2204,17882,414,105,85456,1,0,⋯,9,17,11,10,16,21,15,7,Not measured,MP 3
atac_MUV1#GTGTCAAGTATCGCGC-1,106030,3618,28,1759,15555,219,97,84754,1,0,⋯,23,17,17,23,13,21,17,37,Not measured,MP 2
atac_MUV1#CAAAGCTCAGCGTGAA-1,110002,5461,21,1919,15981,1961,61,84598,1,0,⋯,1,1,3,6,16,26,15,18,Not measured,MP 3
atac_MUV1#ATCCTGCCATTCGTCC-1,105518,3587,17,1751,17431,43,154,82535,1,0,⋯,23,17,17,23,28,21,29,37,Not measured,MP 2
atac_MUV1#TCCAGAACACACATGT-1,108735,4566,15,1938,19875,48,211,82082,1,0,⋯,1,1,3,6,12,26,29,18,Not measured,MP 3
atac_MUV1#GAGATTCCACGTTGTA-1,104994,4461,27,2027,16157,1106,84,81132,1,0,⋯,23,17,17,23,28,21,29,37,Not measured,MP 2


In [ ]:
proj$MP <- reordered_meta$MP

## Export

In [ ]:
proj_name <- 'ArchR_malignant_seacells'

In [ ]:
dir.create(sprintf("%s/export", proj_name))
write.csv(getReducedDims(proj), sprintf('%s/export/svd.csv', proj_name), quote=FALSE)
write.csv(getCellColData(proj), sprintf('%s/export/cell_metadata.csv', proj_name), quote=FALSE)

In [ ]:
# Gene scores
gene.scores <- getMatrixFromProject(proj)
scores <- assays(gene.scores)[['GeneScoreMatrix']]
scores <- as.matrix(scores)
rownames(scores) <- rowData(gene.scores)$name
write.csv(scores, sprintf('%s/export/gene_scores.csv', proj_name), quote=FALSE)

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-2adf50faf8df-Date-2024-08-22_Time-16-21-02.856503.log
If there is an issue, please report to github with logFile!

2024-08-22 16:22:11.104338 : Organizing colData, 1.137 mins elapsed.

2024-08-22 16:22:12.823112 : Organizing rowData, 1.166 mins elapsed.

2024-08-22 16:22:12.83337 : Organizing rowRanges, 1.166 mins elapsed.

2024-08-22 16:22:12.839598 : Organizing Assays (1 of 1), 1.166 mins elapsed.

2024-08-22 16:22:29.839011 : Constructing SummarizedExperiment, 1.45 mins elapsed.

2024-08-22 16:22:30.683867 : Finished Matrix Creation, 1.464 mins elapsed.

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 15.6 GiB”


In [ ]:
# Peak counts
peaks <- getPeakSet(proj)
peak.counts <- getMatrixFromProject(proj, 'PeakMatrix')

# Reorder peaks
# Chromosome order
chr_order <- sort(seqlevels(peaks))
reordered_features <- list()
for(chr in chr_order)
    reordered_features[[chr]] = peaks[seqnames(peaks) == chr]
reordered_features <- Reduce("c", reordered_features)

# Export counts
dir.create(sprintf("%s/export/peak_counts", proj_name))
counts <- assays(peak.counts)[['PeakMatrix']]
writeMM(counts, sprintf('%s/export/peak_counts/counts.mtx', proj_name))
write.csv(colnames(peak.counts), sprintf('%s/export/peak_counts/cells.csv', proj_name), quote=FALSE)
names(reordered_features) <- sprintf("Peak%d", 1:length(reordered_features))
write.csv(as.data.frame(reordered_features), sprintf('%s/export/peak_counts/peaks.csv', proj_name), quote=FALSE)

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-2adf727c6f26-Date-2024-08-22_Time-16-44-42.0764.log
If there is an issue, please report to github with logFile!

2024-08-22 16:45:56.592337 : Organizing colData, 1.242 mins elapsed.

2024-08-22 16:45:57.394065 : Organizing rowData, 1.255 mins elapsed.

2024-08-22 16:45:57.421411 : Organizing rowRanges, 1.256 mins elapsed.

2024-08-22 16:45:57.451494 : Organizing Assays (1 of 1), 1.256 mins elapsed.

2024-08-22 16:46:18.237191 : Constructing SummarizedExperiment, 1.603 mins elapsed.

2024-08-22 16:46:19.539053 : Finished Matrix Creation, 1.624 mins elapsed.



NULL